## 1. Drive and repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!mv /content/OpenPCDet /content/OpenPCDet_partial
!git clone https://github.com/open-mmlab/OpenPCDet.git /content/OpenPCDet
!ls /content/OpenPCDet/tools | head

Cloning into '/content/OpenPCDet'...
remote: Enumerating objects: 4175, done.
remote: Total 4175 (delta 0), reused 0 (delta 0), pack-reused 4175 (from 1)
Receiving objects: 100% (4175/4175), 4.30 MiB | 10.35 MiB/s, done.
Resolving deltas: 100% (2394/2394), done.
cfgs
demo.py
eval_utils
_init_path.py
process_tools
scripts
test.py
train.py
train_utils
visual_utils


In [7]:
!mkdir -p /content/OpenPCDet/data
!mv /content/OpenPCDet_partial/data/kitti /content/OpenPCDet/data/ 2>/dev/null; ls /content/OpenPCDet/data/kitti

ImageSets


In [8]:
!mkdir -p /content/OpenPCDet/data
!mv /content/OpenPCDet_partial/data/kitti /content/OpenPCDet/data/ 2>/dev/null; ls /content/OpenPCDet/data/kitti

ImageSets


In [5]:
!git clone https://github.com/open-mmlab/OpenPCDet.git /content/OpenPCDet

fatal: destination path '/content/OpenPCDet' already exists and is not an empty directory.


## 2. Environment (Python 3.10 + CUDA 11.8 toolkit + gcc 11)

In [ ]:
%cd /content
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba create -y -p /content/pcdet-env \
    -c nvidia/label/cuda-11.8.0 -c conda-forge \
    python=3.10 cuda-toolkit=11.8 "gxx_linux-64=11" "numpy<2" pip

In [ ]:
# PyTorch / spconv / OpenPCDet requirements
!/content/pcdet-env/bin/pip install torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118
!/content/pcdet-env/bin/pip install spconv-cu118 "numpy<2"
!/content/pcdet-env/bin/pip install -r /content/OpenPCDet/requirements.txt "numpy<2"
!/content/pcdet-env/bin/pip install "setuptools==69.5.1" wheel      # torch 2.1.2's build helper needs pkg_resources

In [ ]:
# OpenPCDet imports every dataset module at startup (including Argoverse 2), so these are needed even for KITTI.
# --no-deps keeps pip from replacing the CUDA build of PyTorch.
!/content/pcdet-env/bin/pip install av2 universal-pathlib pathlib-abc fsspec "kornia==0.6.12" --no-deps
!/content/pcdet-env/bin/pip install pandas pyarrow "numpy<2"
!/content/pcdet-env/bin/pip install "numba==0.58.1" "llvmlite==0.41.1" --no-deps
!/content/pcdet-env/bin/python -c "import torch, spconv, kornia, numba; print(torch.__version__, torch.cuda.is_available(), kornia.__version__, numba.__version__)"
# expected: 2.1.2+cu118 True 0.6.12 0.58.1

## 3. Build OpenPCDet's CUDA ops

In [ ]:
import os
env = "/content/pcdet-env"
os.environ["CUDA_HOME"] = env
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"          # T4
os.environ["PATH"] = f"{env}/bin:" + os.environ["PATH"]
os.environ["CC"]  = f"{env}/bin/x86_64-conda-linux-gnu-gcc"    # gcc 11: CUDA 11.8 rejects Colab's system gcc 13
os.environ["CXX"] = f"{env}/bin/x86_64-conda-linux-gnu-g++"

%cd /content/OpenPCDet
!rm -rf build
!/content/pcdet-env/bin/python setup.py develop > /content/build.log 2>&1
!tail -3 /content/build.log
!grep -n -i -E "error|FAILED" /content/build.log | head -20
!/content/pcdet-env/bin/python -c "import torch; import pcdet; from pcdet.ops.iou3d_nms import iou3d_nms_cuda; print('pcdet ok')"

## 4. CPU rotated-IoU for the KITTI evaluation

`numba.cuda` segfaults on this Colab runtime (driver 580 / CUDA 13) when the KITTI evaluator compiles its GPU IoU kernel,
so the AP step crashed. This swaps in a CPU version with the same function name.Predictions still run on the GPU.

In [ ]:
%%writefile /content/rotate_iou_cpu.py
import math
import numba
import numpy as np

@numba.njit(cache=True)
def _corners(box, out):
    a_cos = math.cos(box[4]); a_sin = math.sin(box[4])
    xd = box[2]; yd = box[3]
    cx = (-xd / 2, -xd / 2, xd / 2, xd / 2)
    cy = (-yd / 2, yd / 2, yd / 2, -yd / 2)
    for i in range(4):
        out[i, 0] = a_cos * cx[i] + a_sin * cy[i] + box[0]
        out[i, 1] = -a_sin * cx[i] + a_cos * cy[i] + box[1]

@numba.njit(cache=True)
def _poly_area(p, n):
    s = 0.0
    for i in range(n):
        j = (i + 1) % n
        s += p[i, 0] * p[j, 1] - p[j, 0] * p[i, 1]
    return abs(s) / 2.0

@numba.njit(cache=True)
def _inter_area(b1, b2):
    subj = np.empty((16, 2)); clip = np.empty((4, 2)); tmp = np.empty((16, 2))
    _corners(b1, subj); _corners(b2, clip)
    n = 4
    sgn = 0.0
    for i in range(4):
        j = (i + 1) % 4
        sgn += clip[i, 0] * clip[j, 1] - clip[j, 0] * clip[i, 1]
    sgn = 1.0 if sgn >= 0 else -1.0
    for e in range(4):
        ax, ay = clip[e, 0], clip[e, 1]
        bx, by = clip[(e + 1) % 4, 0], clip[(e + 1) % 4, 1]
        m = 0
        for i in range(n):
            px, py = subj[i, 0], subj[i, 1]
            qx, qy = subj[(i + 1) % n, 0], subj[(i + 1) % n, 1]
            dp = sgn * ((bx - ax) * (py - ay) - (by - ay) * (px - ax))
            dq = sgn * ((bx - ax) * (qy - ay) - (by - ay) * (qx - ax))
            if dp >= 0:
                tmp[m, 0] = px; tmp[m, 1] = py; m += 1
            if (dp >= 0) != (dq >= 0):
                t = dp / (dp - dq)
                tmp[m, 0] = px + t * (qx - px); tmp[m, 1] = py + t * (qy - py); m += 1
        n = m
        if n == 0:
            return 0.0
        for i in range(n):
            subj[i, 0] = tmp[i, 0]; subj[i, 1] = tmp[i, 1]
    if n < 3:
        return 0.0
    return _poly_area(subj, n)

@numba.njit(parallel=True, cache=True)
def _iou_matrix(boxes, qboxes, criterion):
    N = boxes.shape[0]; K = qboxes.shape[0]
    out = np.zeros((N, K), dtype=np.float64)
    for n in numba.prange(N):
        for k in range(K):
            area1 = qboxes[k, 2] * qboxes[k, 3]
            area2 = boxes[n, 2] * boxes[n, 3]
            inter = _inter_area(qboxes[k], boxes[n])
            if criterion == -1:
                out[n, k] = inter / (area1 + area2 - inter)
            elif criterion == 0:
                out[n, k] = inter / area1
            elif criterion == 1:
                out[n, k] = inter / area2
            else:
                out[n, k] = inter
    return out

def rotate_iou_gpu_eval(boxes, query_boxes, criterion=-1, device_id=0):
    """CPU version, same interface as the numba.cuda original."""
    dtype = boxes.dtype
    N, K = boxes.shape[0], query_boxes.shape[0]
    if N == 0 or K == 0:
        return np.zeros((N, K), dtype=dtype)
    b = np.ascontiguousarray(boxes, dtype=np.float64)
    q = np.ascontiguousarray(query_boxes, dtype=np.float64)
    return _iou_matrix(b, q, int(criterion)).astype(dtype)

In [ ]:
D = "/content/OpenPCDet/pcdet/datasets/kitti/kitti_object_eval_python"
!cp -n {D}/rotate_iou.py {D}/rotate_iou_cuda_original.py
!cp /content/rotate_iou_cpu.py {D}/rotate_iou.py

## 5. KITTI data


In [9]:
%%bash
Z=/content/drive/MyDrive/kitti/zips
mkdir -p /content/OpenPCDet/data/kitti
for z in calib label data_object_velodyne image2; do
  echo "extracting $z"
  unzip -q -n "$Z/$z.zip" "training/*" -d /content/OpenPCDet/data/kitti
done
cd /content/OpenPCDet/data/kitti/training
for d in velodyne image_2 label_2 calib; do echo "$d: $(ls $d | wc -l)"; done     # each should be 7481
for s in train val; do echo "$s.txt: $(grep -c '' /content/OpenPCDet/data/kitti/ImageSets/$s.txt)"; done   # 3712 and 3769

extracting calib
extracting label
extracting data_object_velodyne
extracting image2
velodyne: 7481
image_2: 7481
label_2: 7481
calib: 7481
train.txt: 3712
val.txt: 3769


In [4]:
!find /content/drive/MyDrive -name "kitti_infos_val.pkl" 2>/dev/null

/content/drive/MyDrive/kitti_infos/kitti_infos_val.pkl


## 6. Info files


In [3]:
%%writefile /content/OpenPCDet/tools/make_kitti_infos.py
import pickle, yaml
from pathlib import Path
from easydict import EasyDict
import torch  # first, so the compiled ops can find libc10
from pcdet.datasets.kitti.kitti_dataset import KittiDataset

root = Path('/content/OpenPCDet')
data = root / 'data' / 'kitti'
cfg = EasyDict(yaml.safe_load(open(root / 'tools/cfgs/dataset_configs/kitti_dataset.yaml')))
ds = KittiDataset(dataset_cfg=cfg, class_names=['Car', 'Pedestrian', 'Cyclist'], root_path=data, training=False)

infos = {}
for split in ('train', 'val'):
    ds.set_split(split)
    infos[split] = ds.get_infos(num_workers=4, has_label=True, count_inside_pts=True)
    with open(data / f'kitti_infos_{split}.pkl', 'wb') as f:
        pickle.dump(infos[split], f)
    print(split, len(infos[split]), 'frames')
with open(data / 'kitti_infos_trainval.pkl', 'wb') as f:
    pickle.dump(infos['train'] + infos['val'], f)

Writing /content/OpenPCDet/tools/make_kitti_infos.py


FileNotFoundError: [Errno 2] No such file or directory: '/content/OpenPCDet/tools/make_kitti_infos.py'

In [ ]:
%%bash
D=/content/drive/MyDrive/kitti_infos
K=/content/OpenPCDet/data/kitti
if [ -f $D/kitti_infos_train.pkl ] && [ -f $D/kitti_infos_val.pkl ]; then
  cp $D/kitti_infos_*.pkl $K/ && echo "restored info files from Drive"
else
  /content/pcdet-env/bin/python /content/OpenPCDet/tools/make_kitti_infos.py
  mkdir -p $D && cp $K/kitti_infos_*.pkl $D/ && echo "saved info files to Drive"
fi
ls -lh $K/*.pkl

## 7. Pretrained PointPillars checkpoint

From the OpenPCDet README model zoo (KITTI, PointPillar, 18 MB).

In [ ]:
%%bash
C=/content/ckpt/pointpillar_7728.pth
S=/content/drive/MyDrive/pcdet_results/pointpillar_7728.pth
mkdir -p /content/ckpt /content/drive/MyDrive/pcdet_results
if [ -f $S ]; then
  cp $S $C
else
  /content/pcdet-env/bin/pip install -q gdown
  /content/pcdet-env/bin/gdown 1wMxWTpU1qUoY3DsCH31WJmvJxcjFXKlm -O $C     # file ID; recent gdown versions no longer have --fuzzy
  cp $C $S
fi
ls -lh $C

## 8. Baseline evaluation

Runs inference on the 3,769 KITTI val frames and prints AP at the end. Reference values (README, R11, Moderate 3D AP):
Car 77.28, Pedestrian 52.29, Cyclist 62.68.

In [ ]:
%cd /content/OpenPCDet/tools
!/content/pcdet-env/bin/python test.py --cfg_file cfgs/kitti_models/pointpillar.yaml \
    --batch_size 4 --ckpt /content/ckpt/pointpillar_7728.pth

## 9. Runtime per frame

In [ ]:
%cd /content/OpenPCDet/tools
!/content/pcdet-env/bin/python test.py --cfg_file cfgs/kitti_models/pointpillar.yaml \
    --batch_size 1 --ckpt /content/ckpt/pointpillar_7728.pth --infer_time --eval_tag timing 2>&1 | tee /content/timing_run.log

In [ ]:
%%bash
LINE=$(tr '\r' '\n' < /content/timing_run.log | grep -o 'infer_time=[0-9.]*([0-9.]*)' | tail -1)
AVG=$(echo "$LINE" | sed 's/.*(\(.*\))/\1/')
echo "$LINE   (last batch(average) in ms)"
awk -v a="$AVG" 'BEGIN{printf "model forward: %.2f ms/frame = %.1f FPS\n", a, 1000/a}'
nvidia-smi --query-gpu=name --format=csv,noheader

## 10. Re-evaluate saved predictions (optional)

Computes AP from a saved `result.pkl` without re-running the model.

In [ ]:
%%writefile /content/OpenPCDet/tools/eval_from_pkl.py
import sys, pickle, copy
import torch  # first, so the compiled ops can find libc10
from pcdet.datasets.kitti.kitti_object_eval_python import eval as kitti_eval

pred_pkl = sys.argv[1]
info_pkl = '../data/kitti/kitti_infos_val.pkl'
class_names = ['Car', 'Pedestrian', 'Cyclist']

dets = pickle.load(open(pred_pkl, 'rb'))
infos = pickle.load(open(info_pkl, 'rb'))
gts = [copy.deepcopy(i['annos']) for i in infos]

assert len(dets) == len(gts), (len(dets), len(gts))
assert all(d['frame_id'] == i['point_cloud']['lidar_idx'] for d, i in zip(dets, infos)), 'frame order mismatch'

result_str, result_dict = kitti_eval.get_official_eval_result(gts, copy.deepcopy(dets), class_names)
print(result_str)

In [ ]:
%cd /content/OpenPCDet/tools
!/content/pcdet-env/bin/python eval_from_pkl.py ../output/kitti_models/pointpillar/default/eval/epoch_7728/val/default/result.pkl

## 11. Adjustment: robustness to degraded point clouds


In [ ]:
!/content/pcdet-env/bin/pip install -q matplotlib pandas "numpy<2"

In [ ]:
%%writefile /content/OpenPCDet/tools/kitti_ap.py
"""KITTI 3D AP / recall with ignore masks, built on OpenPCDet's own evaluator (OpenPCDet source is not modified).

Restricting an evaluation to a subset of objects (a distance band, a points-per-object bin) uses the same mechanism
KITTI uses for difficulty levels:
  * ground truth outside the subset gets occluded=3 (> every difficulty's limit), so it is "ignored": it is neither a
    miss nor does a detection matched to it count as a false positive;
  * detections outside the subset get a zero-height 2D box, so KITTI's minimum-height rule ignores them: an unmatched
    one is not a false positive.
Only 3D AP at the official thresholds (Car 0.7, Pedestrian 0.5, Cyclist 0.5) is computed, which is ~6x cheaper than
the full report. The 3D IoU matrix does not depend on the masks, so it is computed once and reused.
"""
import contextlib
import numpy as np
try:
    import torch  # noqa: F401  must come before pcdet's compiled ops (libc10); absent only in torch-free unit tests
except ImportError:
    pass
from pcdet.datasets.kitti.kitti_object_eval_python import eval as kitti_eval

CLASSES = ('Car', 'Pedestrian', 'Cyclist')
CLASS_ID = {'Car': 0, 'Pedestrian': 1, 'Cyclist': 2}
IOU_3D = {'Car': 0.7, 'Pedestrian': 0.5, 'Cyclist': 0.5}
DIFFICULTIES = ('easy', 'moderate', 'hard')


def distance(anno):
    """Horizontal distance (m) from the sensor, from camera-frame locations (x right, z forward)."""
    loc = np.asarray(anno['location'], dtype=np.float64).reshape(-1, 3)
    return np.hypot(loc[:, 0], loc[:, 2])


def mask_gt(gts, keep_fn):
    """keep_fn(anno) -> bool array over that frame's ground-truth objects. Others become ignored."""
    out = []
    for a in gts:
        a = dict(a)
        keep = np.asarray(keep_fn(a), dtype=bool)
        occ = np.array(a['occluded'], dtype=np.float64)
        occ[~keep] = 3
        a['occluded'] = occ
        out.append(a)
    return out


def mask_dt(dts, keep_fn):
    out = []
    for a in dts:
        a = dict(a)
        keep = np.asarray(keep_fn(a), dtype=bool)
        bbox = np.array(a['bbox'], dtype=np.float64)
        bbox[~keep, 3] = bbox[~keep, 1]          # zero height -> ignored by KITTI's minimum-height rule
        a['bbox'] = bbox
        out.append(a)
    return out


@contextlib.contextmanager
def cached_overlaps():
    """Compute the IoU matrices once and reuse them for every masked evaluation inside the block.
    All evaluations inside one block must use the same detections and ground-truth geometry (only the masks change)."""
    original = kitti_eval.calculate_iou_partly
    store = {}

    def cached(a, b, metric, num_parts=50):
        if metric not in store:
            store[metric] = original(a, b, metric, num_parts)
        return store[metric]

    kitti_eval.calculate_iou_partly = cached
    try:
        yield
    finally:
        kitti_eval.calculate_iou_partly = original


def ap3d(gts, dts, classes=CLASSES):
    """3D AP at the official thresholds. Returns {class: {'R11': {'easy':..,'moderate':..,'hard':..}, 'R40': {...}}}."""
    ids = [CLASS_ID[c] for c in classes]
    min_overlaps = np.zeros((1, 3, len(ids)))
    for k, c in enumerate(classes):
        min_overlaps[0, :, k] = IOU_3D[c]
    ret = kitti_eval.eval_class(gts, dts, ids, [0, 1, 2], 2, min_overlaps, False)
    prec = ret['precision'][:, :, 0, :]                    # [class, difficulty, 41 points]
    r11, r40 = kitti_eval.get_mAP(prec), kitti_eval.get_mAP_R40(prec)
    return {c: {'R11': dict(zip(DIFFICULTIES, map(float, r11[k]))),
                'R40': dict(zip(DIFFICULTIES, map(float, r40[k])))} for k, c in enumerate(classes)}


def moderate_valid(anno, cls):
    """Ground truth of class `cls` that KITTI's Moderate difficulty counts (same rules as eval.clean_data)."""
    name = np.asarray(anno['name'])
    h = np.asarray(anno['bbox'])[:, 3] - np.asarray(anno['bbox'])[:, 1]
    return ((name == cls) & (np.asarray(anno['occluded']) <= 1) & (np.asarray(anno['truncated']) <= 0.3) & (h > 25))


def gt_recall_table(gts, dts, classes=CLASSES, min_score=0.0):
    """One row per Moderate-valid ground-truth object: class, distance, points inside, best 3D IoU with a same-class
    detection (score >= min_score), and whether it counts as found at the official IoU. Used for recall by distance
    and by points-per-object, which AP cannot split cleanly (false positives do not belong to a bin)."""
    rows = {'cls': [], 'dist': [], 'npts': [], 'iou': [], 'frame': []}
    for fi, (g, d) in enumerate(zip(gts, dts)):
        for cls in classes:
            sel = np.where(moderate_valid(g, cls))[0]
            if len(sel) == 0:
                continue
            gb = np.concatenate([np.asarray(g['location'])[sel], np.asarray(g['dimensions'])[sel],
                                 np.asarray(g['rotation_y'])[sel][:, None]], 1).astype(np.float64)
            dsel = np.where((np.asarray(d['name']) == cls) & (np.asarray(d['score']) >= min_score))[0]
            if len(dsel):
                db = np.concatenate([np.asarray(d['location'])[dsel], np.asarray(d['dimensions'])[dsel],
                                     np.asarray(d['rotation_y'])[dsel][:, None]], 1).astype(np.float64)
                best = kitti_eval.d3_box_overlap(gb, db).max(axis=1)
            else:
                best = np.zeros(len(sel))
            npts = np.asarray(g.get('num_points_in_gt', -np.ones(len(g['name']))))[sel]
            rows['cls'] += [cls] * len(sel)
            rows['dist'] += distance(g)[sel].tolist()
            rows['npts'] += npts.tolist()
            rows['iou'] += best.tolist()
            rows['frame'] += [fi] * len(sel)
    return {k: np.asarray(v) for k, v in rows.items()}

In [ ]:
%%writefile /content/OpenPCDet/tools/perturb.py
"""Point-cloud perturbations for the robustness study (plain numpy; no OpenPCDet imports).

Each op takes (points, rng, **params) and returns new points. Points are KITTI velodyne rows (x, y, z, intensity) in
the LiDAR frame (x forward, y left, z up, sensor at the origin). A frame's random numbers come from
np.random.default_rng([seed, frame_number]), so every run, worker and visualization sees the same perturbed frame.
"""
import numpy as np

# Nominal Velodyne HDL-64E vertical layout (degrees): upper block 32 beams ~1/3 deg apart, lower block 32 beams ~1/2 deg.
# KITTI .bin files carry no ring index, so beams are recovered from elevation angle, which makes `beam_downsample`
# an approximation of dropping every other physical beam.
HDL64_ANGLES = np.concatenate([np.linspace(2.0, -8.33, 32), np.linspace(-8.83, -24.33, 32)])   # descending


def random_drop(points, rng, p):
    """Drop each point independently with probability p (cheaper/sparser sensor, or occlusion by rain/dust)."""
    return points[rng.random(len(points)) >= p]


def beam_downsample(points, rng, keep_every=2):
    """Keep every `keep_every`-th beam (2 -> roughly a 32-beam sensor), beams assigned by nearest nominal elevation."""
    elev = np.degrees(np.arctan2(points[:, 2], np.hypot(points[:, 0], points[:, 1])))
    asc = HDL64_ANGLES[::-1]                                             # ascending for searchsorted
    j = np.clip(np.searchsorted(asc, elev), 1, len(asc) - 1)
    nearest = np.where(np.abs(asc[j] - elev) < np.abs(asc[j - 1] - elev), j, j - 1)
    beam = len(asc) - 1 - nearest                                        # 0 = top beam
    return points[beam % keep_every == 0]


def range_noise(points, rng, sigma):
    """Gaussian noise (std sigma metres) added along each point's ray, as in range-measurement noise."""
    out = points.copy()
    r = np.linalg.norm(out[:, :3], axis=1, keepdims=True)
    r_safe = np.maximum(r, 1e-6)
    out[:, :3] *= (r + rng.normal(0.0, sigma, r.shape)) / r_safe
    return out


def clutter(points, rng, frac, r_range=(3.0, 40.0), z_range=(-1.7, 0.5), half_fov_deg=40.0):
    """Add spurious returns (rain, snow, dust): `frac` x the number of points in the forward camera sector, scattered
    uniformly in range and height inside that sector, with low intensity."""
    az = np.degrees(np.arctan2(points[:, 1], points[:, 0]))
    n_forward = int(np.sum((np.abs(az) < half_fov_deg) & (points[:, 0] > 0)))
    n = int(round(frac * n_forward))
    r = rng.uniform(*r_range, n)
    a = np.radians(rng.uniform(-half_fov_deg, half_fov_deg, n))
    extra = np.stack([r * np.cos(a), r * np.sin(a), rng.uniform(*z_range, n), rng.uniform(0.0, 0.3, n)], axis=1)
    return np.concatenate([points, extra.astype(points.dtype)], axis=0)


OPS = {'random_drop': random_drop, 'beam_downsample': beam_downsample, 'range_noise': range_noise, 'clutter': clutter}

# name -> list of (op, params); applied in order. Edit or add entries freely.
CONDITIONS = {
    'clean':      [],
    'drop50':     [('random_drop', {'p': 0.5})],
    'drop75':     [('random_drop', {'p': 0.75})],
    'beams_half': [('beam_downsample', {'keep_every': 2})],
    'noise_0.1m': [('range_noise', {'sigma': 0.1})],
    'noise_0.3m': [('range_noise', {'sigma': 0.3})],
    'clutter_20': [('clutter', {'frac': 0.2})],
}
DESCRIPTIONS = {
    'clean': 'unmodified', 'drop50': 'random 50% of points removed', 'drop75': 'random 75% of points removed',
    'beams_half': 'every other beam (~32-beam sensor, approximate)', 'noise_0.1m': 'range noise sigma 0.1 m',
    'noise_0.3m': 'range noise sigma 0.3 m', 'clutter_20': 'spurious returns, +20% points in the forward sector',
}


def apply(points, ops, rng):
    for name, params in ops:
        points = OPS[name](points, rng, **params)
    return np.ascontiguousarray(points, dtype=np.float32)


def perturb_frame(points, condition, frame_id, seed=0):
    """Deterministically perturb one frame (frame_id like '000123')."""
    return apply(points, CONDITIONS[condition], np.random.default_rng([seed, int(frame_id)]))


# ---------------------------------------------------------------------------------------------------------------------
# Random corruptions for fine-tuning (continuous severities, so the evaluation levels above are not literally the
# training levels, but the corruption types are the same unless a type is held out with `hold_out`).
TRAIN_OPS = ('random_drop', 'beam_downsample', 'range_noise', 'clutter')


def sample_training_ops(rng, p_clean=0.2, p_two=0.3, hold_out=()):
    """Random list of (op, params) for one training frame. With prob p_clean: no corruption. Otherwise one corruption,
    or (prob p_two) two different ones. Ops named in `hold_out` are never used, to test generalization to an unseen type."""
    if rng.random() < p_clean:
        return []
    names = [n for n in TRAIN_OPS if n not in hold_out]
    k = 2 if (len(names) > 1 and rng.random() < p_two) else 1
    chosen = list(rng.choice(names, size=k, replace=False))
    params = {'random_drop': lambda: {'p': float(rng.uniform(0.2, 0.85))},
              'beam_downsample': lambda: {'keep_every': 2},
              'range_noise': lambda: {'sigma': float(rng.uniform(0.05, 0.35))},
              'clutter': lambda: {'frac': float(rng.uniform(0.05, 0.3))}}
    return [(n, params[n]()) for n in chosen]

In [ ]:
%%writefile /content/OpenPCDet/tools/run_conditions.py
"""Run the pretrained OpenPCDet detector on perturbed KITTI val point clouds.

For every condition in perturb.CONDITIONS it saves, under <out>/<condition>/:
  result.pkl   detections in KITTI annotation format (same layout test.py writes)
  timing.json  per-frame model latency (explicit CUDA sync, batch size 1), CPU preprocessing time, peak GPU memory,
               points and pillars per frame
Finished conditions are skipped, so a Colab disconnect only costs the condition that was running.

  python run_conditions.py --ckpt /content/ckpt/pointpillar_7728.pth --out /content/drive/MyDrive/pcdet_results/robustness
"""
import argparse
import json
import os
import pickle
import time
from pathlib import Path

import numpy as np
import torch
import tqdm
from torch.utils.data import DataLoader

os.chdir(Path(__file__).resolve().parent)          # OpenPCDet configs use paths relative to tools/
import perturb
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets.kitti.kitti_dataset import KittiDataset
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils


class PerturbedKitti(KittiDataset):
    """KittiDataset whose raw point clouds are perturbed before the camera-FOV crop and voxelization."""

    def __init__(self, *args, ops=None, seed=0, **kwargs):
        super().__init__(*args, **kwargs)
        self.ops, self.seed = ops or [], seed

    def get_lidar(self, idx):
        points = super().get_lidar(idx)
        if not self.ops:
            return points
        return perturb.apply(points, self.ops, np.random.default_rng([self.seed, int(idx)]))


def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def make_dataset(ops, args, logger):
    ds = PerturbedKitti(dataset_cfg=cfg.DATA_CONFIG, class_names=cfg.CLASS_NAMES, root_path=None, training=False,
                        logger=logger, ops=ops, seed=args.seed)
    if args.max_frames:
        ds.kitti_infos = ds.kitti_infos[:args.max_frames]
    return ds


def run_condition(name, model, args, logger, out_dir):
    ds = make_dataset(perturb.CONDITIONS[name], args, logger)

    # CPU preprocessing per frame in the main process: read .bin + perturb + FOV crop + voxelization
    n_prep = min(len(ds), 105)
    for i in range(min(5, n_prep)):
        ds[i]
    t0 = time.perf_counter()
    for i in range(5, n_prep):
        ds[i]
    prep_ms = (time.perf_counter() - t0) / max(n_prep - 5, 1) * 1000

    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=args.workers, pin_memory=True,
                        collate_fn=ds.collate_batch, drop_last=False)
    det_annos, times, n_points, n_pillars = [], [], [], []
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for batch in tqdm.tqdm(loader, desc=name, dynamic_ncols=True):
        load_data_to_gpu(batch)
        n_points.append(int(batch['points'].shape[0]))
        n_pillars.append(int(batch['voxels'].shape[0]))
        cuda_sync()
        t = time.perf_counter()
        with torch.no_grad():
            pred_dicts, _ = model(batch)
        cuda_sync()
        times.append((time.perf_counter() - t) * 1000)
        det_annos += ds.generate_prediction_dicts(batch, pred_dicts, cfg.CLASS_NAMES)

    steady = np.array(times[args.warmup:] if len(times) > args.warmup else times)
    timing = {
        'condition': name, 'description': perturb.DESCRIPTIONS.get(name, ''), 'frames': len(times), 'warmup_frames': args.warmup,
        'forward_ms_mean': float(steady.mean()), 'forward_ms_median': float(np.median(steady)),
        'forward_ms_p95': float(np.percentile(steady, 95)), 'fps_from_mean': float(1000.0 / steady.mean()),
        'prep_ms_mean': float(prep_ms),
        'peak_gpu_mem_mb': float(torch.cuda.max_memory_allocated() / 2 ** 20) if torch.cuda.is_available() else None,
        'points_per_frame_mean': float(np.mean(n_points)), 'pillars_per_frame_mean': float(np.mean(n_pillars)),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'times_ms': [round(t, 3) for t in times],
    }
    cond_dir = out_dir / name
    cond_dir.mkdir(parents=True, exist_ok=True)
    with open(cond_dir / 'timing.json', 'w') as f:
        json.dump(timing, f)
    with open(cond_dir / 'result.pkl', 'wb') as f:          # written last: its presence means the condition is complete
        pickle.dump(det_annos, f)
    logger.info('%s: %d frames, forward %.1f ms (median %.1f, p95 %.1f), prep %.1f ms, %.0f points, %.0f pillars',
                name, len(det_annos), timing['forward_ms_mean'], timing['forward_ms_median'], timing['forward_ms_p95'],
                prep_ms, timing['points_per_frame_mean'], timing['pillars_per_frame_mean'])


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--cfg_file', default='cfgs/kitti_models/pointpillar.yaml')
    p.add_argument('--ckpt', required=True)
    p.add_argument('--out', required=True)
    p.add_argument('--conditions', nargs='+', default=list(perturb.CONDITIONS))
    p.add_argument('--workers', type=int, default=4)
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--max_frames', type=int, default=None, help='first N val frames only (quick test)')
    p.add_argument('--warmup', type=int, default=20, help='frames excluded from the latency statistics')
    p.add_argument('--force', action='store_true', help='recompute conditions that already have results')
    args = p.parse_args()

    cfg_from_yaml_file(args.cfg_file, cfg)
    logger = common_utils.create_logger()
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    model = build_network(model_cfg=cfg.MODEL, num_class=len(cfg.CLASS_NAMES), dataset=make_dataset([], args, logger))
    model.load_params_from_file(filename=args.ckpt, logger=logger, to_cpu=False)
    model.cuda()
    model.eval()

    for name in args.conditions:
        if (out_dir / name / 'result.pkl').exists() and not args.force:
            logger.info('%s: already done, skipping (use --force to redo)', name)
            continue
        run_condition(name, model, args, logger, out_dir)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/OpenPCDet/tools/analyze.py
"""Turn the per-condition detections from run_conditions.py into tables and charts.

  python analyze.py --infos .../kitti_infos_val.pkl --results <out of run_conditions> --out <dir>

Writes into --out: ap_overall.csv, ap_by_distance.csv, recall_by_distance.csv, recall_by_points.csv, runtime.csv,
failure_frames.csv and four PNG charts. All AP values are 3D AP (R40) at the official IoU thresholds
(Car 0.7, Pedestrian/Cyclist 0.5), Moderate difficulty unless a column says otherwise.
"""
import os

os.environ["MPLBACKEND"] = "Agg"   # Colab sets an inline backend that a "!python script.py" subprocess cannot use
import argparse
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import kitti_ap
import perturb

BANDS = {'0-20 m': (0, 20), '20-40 m': (20, 40), '40-70 m': (40, 70.4)}
POINT_BINS = [(1, 20), (20, 50), (50, 150), (150, 10 ** 9)]
POINT_LABELS = ['<20 pts', '20-49 pts', '50-149 pts', '150+ pts']


def load(infos_path, results_dir, conditions):
    with open(infos_path, 'rb') as f:
        infos = pickle.load(f)
    dets, timing = {}, {}
    for c in conditions:
        d = Path(results_dir) / c
        if not (d / 'result.pkl').exists():
            print('skipping', c, '(no result.pkl)')
            continue
        with open(d / 'result.pkl', 'rb') as f:
            dets[c] = pickle.load(f)
        timing[c] = json.load(open(d / 'timing.json'))
    n = min(len(v) for v in dets.values())
    infos = infos[:n]
    for c in dets:
        dets[c] = dets[c][:n]
        assert [x['frame_id'] for x in dets[c]] == [i['point_cloud']['lidar_idx'] for i in infos], 'frame order mismatch'
    gts = [i['annos'] for i in infos]
    return gts, dets, timing


def ap_tables(gts, dets, key='R40'):
    """Overall AP (all difficulties) and Moderate AP per distance band, per condition."""
    overall, banded = [], []
    for c, dt in dets.items():
        with kitti_ap.cached_overlaps():
            res = kitti_ap.ap3d(gts, dt)
            for cls in kitti_ap.CLASSES:
                row = {'condition': c, 'class': cls}
                row.update({d: res[cls][key][d] for d in kitti_ap.DIFFICULTIES})
                overall.append(row)
            for band, (lo, hi) in BANDS.items():
                keep = lambda a, lo=lo, hi=hi: (kitti_ap.distance(a) >= lo) & (kitti_ap.distance(a) < hi)
                r = kitti_ap.ap3d(kitti_ap.mask_gt(gts, keep), kitti_ap.mask_dt(dt, keep))
                for cls in kitti_ap.CLASSES:
                    n = sum(int(kitti_ap.moderate_valid(g, cls)[keep(g)].sum()) for g in gts)
                    banded.append({'condition': c, 'class': cls, 'band': band, 'moderate_AP': r[cls][key]['moderate'],
                                   'n_gt_moderate': n})
    return pd.DataFrame(overall), pd.DataFrame(banded)


def recall_tables(gts, dets, clean='clean', min_score=0.0):
    """Recall (found at the official IoU) by distance band and by points-per-object bin.
    Points-per-object comes from the CLEAN cloud so every condition uses the same objects and bins."""
    by_dist, by_pts = [], []
    for c, dt in dets.items():
        t = kitti_ap.gt_recall_table(gts, dt, min_score=min_score)
        found = t['iou'] >= np.array([kitti_ap.IOU_3D[k] for k in t['cls']])
        for cls in kitti_ap.CLASSES:
            m = t['cls'] == cls
            for band, (lo, hi) in BANDS.items():
                s = m & (t['dist'] >= lo) & (t['dist'] < hi)
                by_dist.append({'condition': c, 'class': cls, 'band': band, 'n': int(s.sum()),
                                'recall': float(found[s].mean()) if s.any() else np.nan})
            for lab, (lo, hi) in zip(POINT_LABELS, POINT_BINS):
                s = m & (t['npts'] >= lo) & (t['npts'] < hi)
                by_pts.append({'condition': c, 'class': cls, 'points_bin': lab, 'n': int(s.sum()),
                               'recall': float(found[s].mean()) if s.any() else np.nan})
    return pd.DataFrame(by_dist), pd.DataFrame(by_pts)


def failure_frames(gts, dets, top=15):
    """Frames with the most missed objects on clean data, and the frames each condition hurts most."""
    miss = {}
    for c, dt in dets.items():
        t = kitti_ap.gt_recall_table(gts, dt)
        found = t['iou'] >= np.array([kitti_ap.IOU_3D[k] for k in t['cls']])
        counts = np.bincount(t['frame'][~found].astype(int), minlength=len(gts))
        miss[c] = counts
    rows = []
    clean = miss['clean']
    for fi in np.argsort(-clean)[:top]:
        rows.append({'kind': 'clean_misses', 'condition': 'clean', 'frame_id': gts_id(gts, dets, fi), 'misses': int(clean[fi]),
                     'extra_misses_vs_clean': 0})
    for c, counts in miss.items():
        if c == 'clean':
            continue
        extra = counts - clean
        for fi in np.argsort(-extra)[:5]:
            rows.append({'kind': 'biggest_loss', 'condition': c, 'frame_id': gts_id(gts, dets, fi), 'misses': int(counts[fi]),
                         'extra_misses_vs_clean': int(extra[fi])})
    return pd.DataFrame(rows)


def gts_id(gts, dets, fi):
    return dets['clean'][fi]['frame_id']


def runtime_table(timing):
    rows = []
    for c, t in timing.items():
        rows.append({'condition': c, 'forward_ms_mean': t['forward_ms_mean'], 'forward_ms_median': t['forward_ms_median'],
                     'forward_ms_p95': t['forward_ms_p95'], 'fps': t['fps_from_mean'], 'prep_ms_cpu': t['prep_ms_mean'],
                     'points_per_frame': t['points_per_frame_mean'], 'pillars_per_frame': t['pillars_per_frame_mean'],
                     'peak_gpu_mem_mb': t['peak_gpu_mem_mb']})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------------------------- charts
INK, INK2, GRID, SURF = '#0b0b0b', '#52514e', '#e6e5e1', '#fcfcfb'
BLUES = ['#9ec5f4', '#3987e5', '#184f95']          # sequential steps for ordered bands (near -> far)
BLUE = '#2a78d6'


def _style(ax):
    ax.set_facecolor(SURF)
    for s in ('top', 'right', 'left'):
        ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_color(GRID)
    ax.tick_params(colors=INK2, length=0)
    ax.grid(axis='y', color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def charts(overall, banded, runtime, by_pts, out):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    conds = list(overall['condition'].unique())
    classes = list(kitti_ap.CLASSES)

    # 1. Moderate 3D AP per condition, one panel per class
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True, facecolor=SURF)
    for ax, cls in zip(axes, classes):
        d = overall[overall['class'] == cls].set_index('condition').loc[conds]
        ax.bar(range(len(conds)), d['moderate'], color=BLUE, width=0.6)
        for i, v in enumerate(d['moderate']):
            ax.text(i, v + 1.5, f'{v:.0f}', ha='center', color=INK, fontsize=9)
        ax.set_xticks(range(len(conds)))
        ax.set_xticklabels(conds, rotation=40, ha='right', color=INK2, fontsize=9)
        ax.set_title(cls, color=INK, loc='left', fontsize=12)
        _style(ax)
    axes[0].set_ylabel('3D AP, Moderate (R40, %)', color=INK2)
    axes[0].set_ylim(0, 105)
    fig.suptitle('Accuracy under point-cloud degradation', color=INK, x=0.01, ha='left', fontsize=13)
    fig.tight_layout()
    fig.savefig(Path(out) / 'chart_ap_by_condition.png', dpi=160, facecolor=SURF)
    plt.close(fig)

    # 2. Moderate AP by distance band (Car) - ordered bands use one hue, light -> dark
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True, facecolor=SURF)
    for ax, cls in zip(axes, classes):
        d = banded[banded['class'] == cls]
        w = 0.26
        for k, band in enumerate(BANDS):
            dd = d[d['band'] == band].set_index('condition').reindex(conds)
            ax.bar(np.arange(len(conds)) + (k - 1) * (w + 0.02), dd['moderate_AP'], width=w, color=BLUES[k], label=band)
        ax.set_xticks(range(len(conds)))
        ax.set_xticklabels(conds, rotation=40, ha='right', color=INK2, fontsize=9)
        ax.set_title(cls, color=INK, loc='left', fontsize=12)
        _style(ax)
    axes[0].set_ylabel('3D AP, Moderate (R40, %)', color=INK2)
    axes[0].set_ylim(0, 105)
    axes[0].legend(frameon=False, labelcolor=INK2, fontsize=9, title='distance', title_fontsize=9)
    fig.suptitle('Accuracy by distance band (bands with few objects are noisy - see ap_by_distance.csv for counts)',
                 color=INK, x=0.01, ha='left', fontsize=12)
    fig.tight_layout()
    fig.savefig(Path(out) / 'chart_ap_by_distance.png', dpi=160, facecolor=SURF)
    plt.close(fig)

    # 3. Recall by points per object, Car, clean vs each condition would be crowded -> clean + two harshest
    fig, ax = plt.subplots(figsize=(7.5, 4.2), facecolor=SURF)
    d = by_pts[by_pts['class'] == 'Car']
    w = 0.26
    show = [c for c in ('clean', 'drop75', 'noise_0.3m') if c in conds]
    shades = ['#184f95', '#3987e5', '#9ec5f4']
    for k, c in enumerate(show):
        dd = d[d['condition'] == c].set_index('points_bin').reindex(POINT_LABELS)
        ax.bar(np.arange(len(POINT_LABELS)) + (k - 1) * (w + 0.02), dd['recall'] * 100, width=w, color=shades[k], label=c)
    ax.set_xticks(range(len(POINT_LABELS)))
    ax.set_xticklabels(POINT_LABELS, color=INK2)
    ax.set_ylabel('Car recall (%)', color=INK2)
    ax.set_xlabel('LiDAR points inside the object in the clean scan', color=INK2)
    ax.legend(frameon=False, labelcolor=INK2, fontsize=9)
    ax.set_title('Sparse objects are lost first', color=INK, loc='left', fontsize=12)
    _style(ax)
    fig.tight_layout()
    fig.savefig(Path(out) / 'chart_recall_by_points.png', dpi=160, facecolor=SURF)
    plt.close(fig)

    # 4. Runtime per condition (median forward pass, p95 as a tick)
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=SURF)
    r = runtime.set_index('condition').loc[conds]
    ax.bar(range(len(conds)), r['forward_ms_median'], color=BLUE, width=0.6)
    ax.scatter(range(len(conds)), r['forward_ms_p95'], color=INK, marker='_', s=300, zorder=3, label='p95')
    for i, v in enumerate(r['forward_ms_median']):
        ax.text(i, v / 2, f'{v:.1f}', ha='center', color='white', fontsize=9)
    ax.set_xticks(range(len(conds)))
    ax.set_xticklabels(conds, rotation=40, ha='right', color=INK2, fontsize=9)
    ax.set_ylabel('Forward pass (ms, batch 1)', color=INK2)
    ax.legend(frameon=False, labelcolor=INK2, fontsize=9)
    ax.set_title('Model latency per frame (median, with p95)', color=INK, loc='left', fontsize=12)
    _style(ax)
    fig.tight_layout()
    fig.savefig(Path(out) / 'chart_runtime.png', dpi=160, facecolor=SURF)
    plt.close(fig)


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--infos', required=True)
    p.add_argument('--results', required=True)
    p.add_argument('--out', required=True)
    p.add_argument('--conditions', nargs='+', default=list(perturb.CONDITIONS))
    a = p.parse_args()
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)

    gts, dets, timing = load(a.infos, a.results, a.conditions)
    overall, banded = ap_tables(gts, dets)
    rdist, rpts = recall_tables(gts, dets)
    runtime = runtime_table(timing)
    fails = failure_frames(gts, dets)
    for name, df in [('ap_overall', overall), ('ap_by_distance', banded), ('recall_by_distance', rdist),
                     ('recall_by_points', rpts), ('runtime', runtime), ('failure_frames', fails)]:
        df.to_csv(out / f'{name}.csv', index=False)

    pd.set_option('display.width', 200, 'display.max_columns', 30, 'display.float_format', '{:.1f}'.format)
    print('\n3D AP (R40, %), all difficulties\n', overall.pivot(index='condition', columns='class', values='moderate')
          .loc[list(dets)].rename(columns=lambda c: c + ' (Mod)'))
    print('\nModerate AP by distance band\n', banded.pivot_table(index='condition', columns=['class', 'band'],
                                                               values='moderate_AP').loc[list(dets)])
    print('\nObjects per band (Moderate GT)\n', banded[banded.condition == 'clean'].pivot(index='band', columns='class',
                                                                                         values='n_gt_moderate'))
    print('\nCar recall by points per object\n', rpts[rpts['class'] == 'Car'].pivot(index='condition', columns='points_bin',
                                                                                 values='recall').loc[list(dets)][POINT_LABELS] * 100)
    print('\nRuntime\n', runtime.set_index('condition'))
    charts(overall, banded, runtime, rpts, out)
    print('\nWrote tables and charts to', out)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/OpenPCDet/tools/visualize.py
"""Bird's-eye-view and 3D figures: point cloud + ground truth + predictions, for any frame and condition.

  from visualize import render
  render('000123', 'clean', infos, results_dir, kitti_root, 'out/figs')              # one condition
  render('000123', ['clean', 'drop75'], ...)                                         # before/after side by side
Ground truth: solid green. Predictions: dashed, coloured by class, labelled with score (threshold --min_score).
Boxes are in the LiDAR frame (x forward, y left, z up), as OpenPCDet's `gt_boxes_lidar` / `boxes_lidar`.
"""
import os

os.environ["MPLBACKEND"] = "Agg"   # Colab sets an inline backend that a "!python script.py" subprocess cannot use
import pickle
from pathlib import Path

import numpy as np

import perturb

CLASS_COLOR = {'Car': '#eb6834', 'Pedestrian': '#6250d6', 'Cyclist': '#1baf7a'}
GT_COLOR = '#008300'
XLIM, YLIM = (0, 70.4), (-40, 40)


def box_corners(b):
    """b = (x, y, z, dx, dy, dz, heading), z at the box centre -> 8x3 corners (bottom 0-3, top 4-7)."""
    x, y, z, dx, dy, dz, h = [float(v) for v in b[:7]]
    c = np.array([[dx, dy], [dx, -dy], [-dx, -dy], [-dx, dy]]) / 2
    R = np.array([[np.cos(h), -np.sin(h)], [np.sin(h), np.cos(h)]])
    xy = c @ R.T + [x, y]
    return np.vstack([np.c_[xy, np.full(4, z - dz / 2)], np.c_[xy, np.full(4, z + dz / 2)]])


def load_points(kitti_root, frame_id, condition, seed=0):
    f = Path(kitti_root) / 'training' / 'velodyne' / f'{frame_id}.bin'
    pts = np.fromfile(f, dtype=np.float32).reshape(-1, 4)
    pts = perturb.perturb_frame(pts, condition, frame_id, seed)
    m = (pts[:, 0] > XLIM[0]) & (pts[:, 0] < XLIM[1]) & (np.abs(pts[:, 1]) < YLIM[1])
    return pts[m]


def frame_boxes(infos, dets, frame_id, min_score):
    info = next(i for i in infos if i['point_cloud']['lidar_idx'] == frame_id)
    gt = np.asarray(info['annos']['gt_boxes_lidar'])
    names = np.asarray(info['annos']['name'])
    gt_names = names[names != 'DontCare']          # gt_boxes_lidar has no DontCare rows, so align the names with it
    assert len(gt_names) == len(gt), (len(gt_names), len(gt))
    ok = np.isin(gt_names, list(CLASS_COLOR))       # draw only the three evaluated classes (not Van, Truck, Misc, ...)
    d = next(x for x in dets if x['frame_id'] == frame_id)
    keep = np.asarray(d['score']) >= min_score
    return gt[ok], gt_names[ok], np.asarray(d['boxes_lidar'])[keep], np.asarray(d['name'])[keep], np.asarray(d['score'])[keep]


def _bev(ax, pts, gt, gt_n, pr, pr_n, pr_s):
    ax.scatter(pts[:, 0], pts[:, 1], s=0.4, c=np.clip(pts[:, 2], -2, 1.5), cmap='Greys', vmin=-2.5, vmax=2, linewidths=0)
    for b in gt:
        c = box_corners(b)[:4]
        ax.plot(*np.r_[c, c[:1]].T[:2], color=GT_COLOR, lw=1.6)
    for b, n, s in zip(pr, pr_n, pr_s):
        c = box_corners(b)[:4]
        ax.plot(*np.r_[c, c[:1]].T[:2], color=CLASS_COLOR.get(n, 'k'), lw=1.4, ls='--')
        ax.text(c[:, 0].max(), c[:, 1].max(), f'{s:.2f}', fontsize=6, color=CLASS_COLOR.get(n, 'k'))
    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM); ax.set_aspect('equal')
    ax.set_xlabel('x forward (m)'); ax.set_ylabel('y left (m)')


EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]


def _view3d(ax, pts, gt, pr, pr_n, elev=22, azim=180):
    sel = (pts[:, 0] < 40) & (np.abs(pts[:, 1]) < 20)
    ax.scatter(pts[sel, 0], pts[sel, 1], pts[sel, 2], s=0.3, c=pts[sel, 2], cmap='viridis', vmin=-2, vmax=1.5, linewidths=0)
    for b in gt:
        c = box_corners(b)
        for i, j in EDGES:
            ax.plot(*c[[i, j]].T, color=GT_COLOR, lw=1.2)
    for b, n in zip(pr, pr_n):
        c = box_corners(b)
        for i, j in EDGES:
            ax.plot(*c[[i, j]].T, color=CLASS_COLOR.get(n, 'k'), lw=1.0, ls='--')
    ax.set_xlim(0, 40); ax.set_ylim(-20, 20); ax.set_zlim(-3, 3)
    ax.set_box_aspect((40, 40, 8))
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')


def render(frame_id, conditions, infos, results_dir, kitti_root, out_dir, min_score=0.3, seed=0, three_d=True):
    """Save <out_dir>/<frame>_bev.png and (optionally) <frame>_3d.png; one column per condition."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    conditions = [conditions] if isinstance(conditions, str) else list(conditions)
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    for kind in (['bev', '3d'] if three_d else ['bev']):
        n = len(conditions)
        fig = plt.figure(figsize=(7.5 * n if kind == 'bev' else 6.5 * n, 5.2 if kind == 'bev' else 5.5))
        for k, cond in enumerate(conditions):
            with open(Path(results_dir) / cond / 'result.pkl', 'rb') as f:
                dets = pickle.load(f)
            pts = load_points(kitti_root, frame_id, cond, seed)
            gt, gt_n, pr, pr_n, pr_s = frame_boxes(infos, dets, frame_id, min_score)
            if kind == 'bev':
                ax = fig.add_subplot(1, n, k + 1)
                _bev(ax, pts, gt, gt_n, pr, pr_n, pr_s)
            else:
                ax = fig.add_subplot(1, n, k + 1, projection='3d')
                _view3d(ax, pts, gt, pr, pr_n)
            ax.set_title(f'{frame_id}  {cond}  ({len(pts):,} pts in view, {len(pr)} detections, {len(gt)} GT)', fontsize=9)
        handles = [plt.Line2D([], [], color=GT_COLOR, lw=1.6, label='ground truth')] + \
                  [plt.Line2D([], [], color=c, ls='--', lw=1.4, label=f'pred {n}') for n, c in CLASS_COLOR.items()]
        fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False, fontsize=8)
        fig.tight_layout(rect=(0, 0.05, 1, 1))
        path = out_dir / f'{frame_id}_{kind}_{"-".join(conditions)}.png'
        fig.savefig(path, dpi=150)
        plt.close(fig)
        saved.append(path)
    return saved


if __name__ == '__main__':
    import argparse
    import pandas as pd
    p = argparse.ArgumentParser()
    p.add_argument('--infos', required=True)
    p.add_argument('--results', required=True)
    p.add_argument('--kitti', required=True, help='KITTI root that contains training/velodyne')
    p.add_argument('--out', required=True)
    p.add_argument('--frames', nargs='*', help='frame ids; default: picked from failure_frames.csv in --analysis')
    p.add_argument('--analysis', help='folder written by analyze.py')
    p.add_argument('--conditions', nargs='+', default=['clean', 'drop75'])
    p.add_argument('--min_score', type=float, default=0.3)
    a = p.parse_args()
    with open(a.infos, 'rb') as f:
        infos = pickle.load(f)
    frames = a.frames
    if not frames:
        ff = pd.read_csv(Path(a.analysis) / 'failure_frames.csv', dtype={'frame_id': str})
        frames = list(dict.fromkeys(ff[ff.kind == 'clean_misses'].frame_id[:2].tolist() +
                                    ff[(ff.kind == 'biggest_loss') & (ff.condition == a.conditions[-1])].frame_id[:2].tolist()))
    for fid in frames:
        for path in render(fid, a.conditions, infos, a.results, a.kitti, a.out, a.min_score):
            print(path)

### 11.1 Run all conditions


In [ ]:
%cd /content/OpenPCDet/tools
!/content/pcdet-env/bin/python run_conditions.py --ckpt /content/ckpt/pointpillar_7728.pth \
    --out /content/drive/MyDrive/pcdet_results/robustness --workers 4

### 11.2 Tables and charts


In [ ]:
%cd /content/OpenPCDet/tools
!/content/pcdet-env/bin/python analyze.py --infos ../data/kitti/kitti_infos_val.pkl \
    --results /content/drive/MyDrive/pcdet_results/robustness \
    --out /content/drive/MyDrive/pcdet_results/robustness_analysis

In [ ]:
from IPython.display import Image, display
A = '/content/drive/MyDrive/pcdet_results/robustness_analysis/'
for n in ['chart_ap_by_condition', 'chart_ap_by_distance', 'chart_recall_by_points', 'chart_runtime']:
    display(Image(A + n + '.png'))

### 11.3 Point clouds, ground truth and predictions (BEV and 3D)

In [ ]:
%cd /content/OpenPCDet/tools
!/content/pcdet-env/bin/python visualize.py --infos ../data/kitti/kitti_infos_val.pkl \
    --results /content/drive/MyDrive/pcdet_results/robustness --kitti ../data/kitti \
    --analysis /content/drive/MyDrive/pcdet_results/robustness_analysis \
    --out /content/drive/MyDrive/pcdet_results/figures --conditions clean drop75

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('/content/drive/MyDrive/pcdet_results/figures/*.png')):
    print(f.split('/')[-1]); display(Image(f))

### 11.4 Failure cases to write up

In [ ]:
import pandas as pd
pd.read_csv('/content/drive/MyDrive/pcdet_results/robustness_analysis/failure_frames.csv', dtype={'frame_id': str})

## 12. Fine-tuning on degraded point clouds (before vs after)


In [ ]:
%%writefile /content/OpenPCDet/tools/finetune.py
"""Fine-tune a pretrained OpenPCDet model on randomly degraded point clouds, using OpenPCDet's own train.py unchanged.

This launcher only (1) patches the training-time DataProcessor so each training frame's points get a random corruption
from perturb.sample_training_ops (after OpenPCDet's own augmentation: GT sampling, flip, rotation, scaling; before
range masking and voxelization), then (2) runs train.main(). Everything else on the command line goes to train.py.

  python finetune.py --hold_out clutter --cfg_file cfgs/kitti_models/pointpillar.yaml --pretrained_model ckpt.pth \
      --epochs 8 --extra_tag finetune_degraded --set OPTIMIZATION.LR 0.0003

If KITTI's road-plane files (training/planes) are absent, USE_ROAD_PLANE is switched off for GT sampling.

Evaluation frames are corrupted before the camera-FOV crop (run_conditions.py); training frames are corrupted after
the FOV crop and augmentation. For per-point corruptions (drop, noise, beams) that is equivalent; for clutter the
spurious points are placed in a +-40 deg sector of the augmented (rotated/flipped) cloud rather than of the raw one.
"""
import argparse
import sys

import numpy as np
import torch  # before pcdet's compiled ops

import perturb


def install(p_clean, hold_out):
    from pcdet.datasets.processor.data_processor import DataProcessor
    original = DataProcessor.forward

    def forward(self, data_dict):
        if self.training and data_dict.get('points') is not None:
            ops = perturb.sample_training_ops(np.random.default_rng(), p_clean=p_clean, hold_out=hold_out)
            if ops:
                data_dict['points'] = perturb.apply(data_dict['points'], ops, np.random.default_rng())
        return original(self, data_dict)

    DataProcessor.forward = forward


if __name__ == '__main__':
    ap = argparse.ArgumentParser(add_help=False)
    ap.add_argument('--p_clean', type=float, default=0.2, help='fraction of training frames left uncorrupted')
    ap.add_argument('--hold_out', nargs='*', default=[], choices=perturb.TRAIN_OPS,
                    help='corruption types never used in training (to test generalization)')
    own, rest = ap.parse_known_args()
    install(own.p_clean, tuple(own.hold_out))
    sys.argv = [sys.argv[0]] + rest
    import train
    from pathlib import Path

    original_parse_config = train.parse_config

    def parse_config_without_missing_planes():
        args, cfg = original_parse_config()
        planes = Path(cfg.DATA_CONFIG.DATA_PATH) / 'training' / 'planes'
        if not planes.exists():          # road planes are a separate KITTI download; without them GT sampling cannot snap to the road
            for aug in cfg.DATA_CONFIG.DATA_AUGMENTOR.AUG_CONFIG_LIST:
                if aug.get('USE_ROAD_PLANE', False):
                    aug['USE_ROAD_PLANE'] = False
                    print(f'finetune.py: {planes} not found, USE_ROAD_PLANE disabled for {aug["NAME"]}')
        return args, cfg

    train.parse_config = parse_config_without_missing_planes
    train.main()

In [ ]:
%%writefile /content/OpenPCDet/tools/make_gt_database.py
"""Create the KITTI ground-truth sampling database (gt_database/ + kitti_dbinfos_train.pkl) that training augmentation
needs. Not needed for evaluation. Takes a few minutes; skipped if it already exists."""
import pickle
from pathlib import Path

import torch  # first, so the compiled ops can find libc10
import yaml
from easydict import EasyDict

from pcdet.datasets.kitti.kitti_dataset import KittiDataset
import pcdet.datasets.kitti.kitti_dataset as kitti_module

kitti_module.Path = Path   # OpenPCDet only imports Path inside its own __main__ block, so create_groundtruth_database fails without this

root = Path('/content/OpenPCDet')
data = root / 'data' / 'kitti'
if (data / 'kitti_dbinfos_train.pkl').exists():
    print('already exists:', data / 'kitti_dbinfos_train.pkl')
else:
    cfg = EasyDict(yaml.safe_load(open(root / 'tools/cfgs/dataset_configs/kitti_dataset.yaml')))
    ds = KittiDataset(dataset_cfg=cfg, class_names=['Car', 'Pedestrian', 'Cyclist'], root_path=data, training=False)
    ds.set_split('train')
    ds.create_groundtruth_database(info_path=data / 'kitti_infos_train.pkl', split='train')
    print('done')

In [ ]:
%%writefile /content/OpenPCDet/tools/compare.py
"""Side-by-side pretrained vs fine-tuned results from two analyze.py output folders.

  python compare.py --a <analysis dir, pretrained> --b <analysis dir, fine-tuned> --out <dir> [--names pretrained finetuned]
Writes compare_ap.csv (Moderate 3D AP and difference per class and condition), compare_distance.csv, chart_compare.png.
"""
import os

os.environ["MPLBACKEND"] = "Agg"   # Colab sets an inline backend that a "!python script.py" subprocess cannot use
import argparse
from pathlib import Path

import pandas as pd

INK, INK2, GRID, SURF = '#0b0b0b', '#52514e', '#e6e5e1', '#fcfcfb'
SERIES = ['#2a78d6', '#eb6834']          # categorical slots 1 and 2


def merge(a, b, keys, value, names):
    m = a.merge(b, on=keys, suffixes=('_a', '_b'))
    out = m[keys].copy()
    out[names[0]] = m[value + '_a']
    out[names[1]] = m[value + '_b']
    out['change'] = out[names[1]] - out[names[0]]
    return out


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--a', required=True)
    p.add_argument('--b', required=True)
    p.add_argument('--out', required=True)
    p.add_argument('--names', nargs=2, default=['pretrained', 'finetuned'])
    args = p.parse_args()
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    A, B = Path(args.a), Path(args.b)
    ap = merge(pd.read_csv(A / 'ap_overall.csv'), pd.read_csv(B / 'ap_overall.csv'), ['condition', 'class'], 'moderate', args.names)
    dist = merge(pd.read_csv(A / 'ap_by_distance.csv'), pd.read_csv(B / 'ap_by_distance.csv'),
                 ['condition', 'class', 'band'], 'moderate_AP', args.names)
    ap.to_csv(out / 'compare_ap.csv', index=False)
    dist.to_csv(out / 'compare_distance.csv', index=False)

    pd.set_option('display.width', 200, 'display.float_format', '{:.1f}'.format)
    for cls in ('Car', 'Pedestrian', 'Cyclist'):
        print(f'\n{cls}: Moderate 3D AP (R40, %)\n', ap[ap['class'] == cls].drop(columns='class').set_index('condition'))

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import numpy as np
    conds = list(ap['condition'].unique())
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.4), sharey=True, facecolor=SURF)
    for ax, cls in zip(axes, ('Car', 'Pedestrian', 'Cyclist')):
        d = ap[ap['class'] == cls].set_index('condition').loc[conds]
        w = 0.38
        for k, n in enumerate(args.names):
            ax.bar(np.arange(len(conds)) + (k - 0.5) * (w + 0.02), d[n], width=w, color=SERIES[k], label=n)
        ax.set_xticks(range(len(conds)))
        ax.set_xticklabels(conds, rotation=40, ha='right', color=INK2, fontsize=9)
        ax.set_title(cls, color=INK, loc='left', fontsize=12)
        ax.set_facecolor(SURF)
        for s in ('top', 'right', 'left'):
            ax.spines[s].set_visible(False)
        ax.spines['bottom'].set_color(GRID)
        ax.tick_params(colors=INK2, length=0)
        ax.grid(axis='y', color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    axes[0].set_ylabel('3D AP, Moderate (R40, %)', color=INK2)
    axes[0].set_ylim(0, 100)
    axes[0].legend(frameon=False, labelcolor=INK2, fontsize=9)
    fig.suptitle('Before vs after fine-tuning on degraded point clouds', color=INK, x=0.01, ha='left', fontsize=13)
    fig.tight_layout()
    fig.savefig(out / 'chart_compare.png', dpi=160, facecolor=SURF)
    print('\nWrote', out)


if __name__ == '__main__':
    main()

In [ ]:
# perturb.py (Section 11) now also holds the random training corruptions; rewrite it if you ran Section 11 with an older copy.
%cd /content/OpenPCDet/tools
!/content/pcdet-env/bin/python make_gt_database.py

### 12.1 Train

In [ ]:
%%bash
D=/content/drive/MyDrive/pcdet_results/finetune_degraded
O=/content/OpenPCDet/output/kitti_models/pointpillar
mkdir -p $D $O
ln -sfn $D $O/finetune_degraded          # train.py writes checkpoints and logs through this link, i.e. into Drive
cd /content/OpenPCDet/tools
PYTHONUNBUFFERED=1 /content/pcdet-env/bin/python -u finetune.py --cfg_file cfgs/kitti_models/pointpillar.yaml \
    --pretrained_model /content/ckpt/pointpillar_7728.pth --epochs 8 --batch_size 4 --workers 2 \
    --extra_tag finetune_degraded --max_ckpt_save_num 3 --num_epochs_to_eval 1 \
    --set OPTIMIZATION.LR 0.0003
ls -lh $D/ckpt

### 12.2 Evaluate the fine-tuned model on all conditions

In [ ]:
%cd /content/OpenPCDet/tools
!/content/pcdet-env/bin/python run_conditions.py \
    --ckpt /content/drive/MyDrive/pcdet_results/finetune_degraded/ckpt/checkpoint_epoch_8.pth \
    --out /content/drive/MyDrive/pcdet_results/robustness_finetuned --workers 4
!/content/pcdet-env/bin/python analyze.py --infos ../data/kitti/kitti_infos_val.pkl \
    --results /content/drive/MyDrive/pcdet_results/robustness_finetuned \
    --out /content/drive/MyDrive/pcdet_results/robustness_finetuned_analysis

### 12.3 Before vs after

In [ ]:
%cd /content/OpenPCDet/tools
!/content/pcdet-env/bin/python compare.py \
    --a /content/drive/MyDrive/pcdet_results/robustness_analysis \
    --b /content/drive/MyDrive/pcdet_results/robustness_finetuned_analysis \
    --out /content/drive/MyDrive/pcdet_results/comparison

In [ ]:
from IPython.display import Image, display
display(Image('/content/drive/MyDrive/pcdet_results/comparison/chart_compare.png'))

### 12.4 Same frames, fine-tuned model

In [ ]:
%cd /content/OpenPCDet/tools
!/content/pcdet-env/bin/python visualize.py --infos ../data/kitti/kitti_infos_val.pkl \
    --results /content/drive/MyDrive/pcdet_results/robustness_finetuned --kitti ../data/kitti \
    --analysis /content/drive/MyDrive/pcdet_results/robustness_analysis \
    --out /content/drive/MyDrive/pcdet_results/figures_finetuned --conditions clean drop75

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('/content/drive/MyDrive/pcdet_results/figures_finetuned/*.png')):
    print(f.split('/')[-1]); display(Image(f))

## 13. Save results to Drive

In [ ]:
%%bash
R=/content/drive/MyDrive/pcdet_results
mkdir -p $R/baseline $R/code
cp /content/OpenPCDet/output/kitti_models/pointpillar/default/eval/epoch_7728/val/default/result.pkl $R/baseline/
cp /content/timing_run.log $R/baseline/ 2>/dev/null || true
cp /content/OpenPCDet/tools/eval_from_pkl.py /content/OpenPCDet/tools/make_kitti_infos.py /content/rotate_iou_cpu.py $R/code/
T=/content/OpenPCDet/tools
cp $T/kitti_ap.py $T/perturb.py $T/run_conditions.py $T/analyze.py $T/visualize.py $T/finetune.py $T/make_gt_database.py $T/compare.py $T/make_demo_video.py $R/code/ 2>/dev/null || true
ls -lh $R/baseline $R/code

## 14. Demo video (at most 1 minute)

In [10]:
%%writefile /content/OpenPCDet/tools/make_demo_video.py
"""Demo video (<= 60 s) from saved predictions: no detector run, no GPU needed.

Per scene: three bird's-eye panels  [pretrained, clean] [pretrained, 75% dropped] [fine-tuned, 75% dropped],
then a rotating 3D view (pretrained clean vs fine-tuned dropped). Ground truth solid green, predictions dashed.
  python make_demo_video.py --infos kitti_infos_val.pkl --kitti /content/kitti --pre <robustness dir> --ft <robustness_finetuned dir> \
         --out demo.mp4 --frames 002477 004527 005510 006682
Note: KITTI object-detection frames are single snapshots, not a continuous drive, so the video steps through chosen frames.
"""
import os

os.environ["MPLBACKEND"] = "Agg"
import pickle
import shutil
import subprocess
import tempfile
from pathlib import Path

import numpy as np

import visualize as V

MAX_SECONDS = 58.0


def _dets(cache, d, cond):
    k = (str(d), cond)
    if k not in cache:
        with open(Path(d) / cond / 'result.pkl', 'rb') as f:
            cache[k] = pickle.load(f)
    return cache[k]


def scene_image(fid, panels, infos, kitti, cache, min_score, path, caption):
    """panels: list of (title, results_dir, condition)."""
    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(12.8, 7.2), dpi=100)
    n = len(panels)
    for k, (title, rdir, cond) in enumerate(panels):
        pts = V.load_points(kitti, fid, cond)
        gt, gt_n, pr, pr_n, pr_s = V.frame_boxes(infos, _dets(cache, rdir, cond), fid, min_score)
        ax = fig.add_subplot(1, n, k + 1)
        V._bev(ax, pts, gt, gt_n, pr, pr_n, pr_s)
        ax.set_title(f'{title}\n{len(pr)} detections / {len(gt)} GT, {len(pts):,} pts', fontsize=10)
    handles = [plt.Line2D([], [], color=V.GT_COLOR, lw=2, label='ground truth')] + \
              [plt.Line2D([], [], color=c, ls='--', lw=1.6, label=f'predicted {n}') for n, c in V.CLASS_COLOR.items()]
    fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False, fontsize=10)
    fig.suptitle(f'Frame {fid}: {caption}', fontsize=13)
    fig.tight_layout(rect=(0, 0.06, 1, 0.95))
    fig.savefig(path)
    plt.close(fig)


def rotating_images(fid, panels, infos, kitti, cache, min_score, out_dir, n_frames):
    import matplotlib.pyplot as plt
    data = []
    for title, rdir, cond in panels:
        pts = V.load_points(kitti, fid, cond)
        gt, _, pr, pr_n, _ = V.frame_boxes(infos, _dets(cache, rdir, cond), fid, min_score)
        data.append((title, pts, gt, pr, pr_n))
    paths = []
    for i in range(n_frames):
        azim = 180 + 360.0 * i / n_frames
        fig = plt.figure(figsize=(12.8, 7.2), dpi=100)
        for k, (title, pts, gt, pr, pr_n) in enumerate(data):
            ax = fig.add_subplot(1, len(data), k + 1, projection='3d')
            V._view3d(ax, pts, gt, pr, pr_n, elev=22, azim=azim)
            ax.set_title(title, fontsize=10)
        fig.suptitle(f'Frame {fid}: rotating 3D view (green = ground truth, dashed = predictions)', fontsize=13)
        p = Path(out_dir) / f'rot_{i:04d}.png'
        fig.savefig(p)
        plt.close(fig)
        paths.append(p)
    return paths


def card(path, lines):
    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(12.8, 7.2), dpi=100)
    fig.text(0.5, 0.58, lines[0], ha='center', fontsize=26, weight='bold')
    for i, t in enumerate(lines[1:]):
        fig.text(0.5, 0.46 - 0.07 * i, t, ha='center', fontsize=15)
    fig.savefig(path)
    plt.close(fig)


def build(infos, kitti, pre, ft, out, frames, rotate_frame=None, min_score=0.3, fps=12, rot_seconds=8.0, ffmpeg='ffmpeg'):
    cache = {}
    tmp = Path(tempfile.mkdtemp(prefix='demo_'))
    title_s, end_s = 3.0, 3.0
    per = max(1.5, min(5.0, (MAX_SECONDS - title_s - end_s - rot_seconds - 0.5) / max(1, len(frames))))
    steps = []     # (image path, seconds)
    card(tmp / 'title.png', ['PointPillars on KITTI (OpenPCDet)', 'Pretrained vs. fine-tuned detector on clean and degraded LiDAR',
                            'Tesla T4, KITTI validation frames, predictions with score >= %.1f' % min_score])
    steps.append((tmp / 'title.png', title_s))
    for fid in frames:
        p = tmp / f'scene_{fid}.png'
        scene_image(fid, [('Pretrained, clean', pre, 'clean'), ('Pretrained, 75% of points dropped', pre, 'drop75'),
                          ('Fine-tuned, 75% of points dropped', ft, 'drop75')], infos, kitti, cache, min_score, p,
                    'same scene under three conditions')
        steps.append((p, per))
    n_rot = int(round(rot_seconds * fps))
    rot = rotating_images(rotate_frame or frames[0], [('Pretrained, clean', pre, 'clean'), ('Fine-tuned, 75% dropped', ft, 'drop75')],
                          infos, kitti, cache, min_score, tmp, n_rot)
    card(tmp / 'end.png', ['Summary', 'Fine-tuning on degraded clouds recovers 8 to 25 AP points on the worst conditions',
                           'at a cost of 1 to 7 points on clean data (single run, synthetic corruptions)'])
    # still scenes -> clips via concat list; rotation as an image sequence
    lst = tmp / 'list.txt'
    with open(lst, 'w') as f:
        for p, s in steps:
            f.write(f"file '{p}'\nduration {s}\n")
        f.write(f"file '{steps[-1][0]}'\n")
    seg1 = tmp / 'seg1.mp4'
    vf = f'fps={fps},scale=1280:720,format=yuv420p'
    run = lambda c: subprocess.run(c, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    run([ffmpeg, '-y', '-f', 'concat', '-safe', '0', '-i', str(lst), '-vf', vf, '-c:v', 'libx264', str(seg1)])
    seg2 = tmp / 'seg2.mp4'
    run([ffmpeg, '-y', '-framerate', str(fps), '-i', str(tmp / 'rot_%04d.png'), '-vf', 'scale=1280:720,format=yuv420p', '-c:v', 'libx264', str(seg2)])
    seg3 = tmp / 'seg3.mp4'
    run([ffmpeg, '-y', '-loop', '1', '-t', str(end_s), '-i', str(tmp / 'end.png'), '-vf', vf, '-c:v', 'libx264', str(seg3)])
    cl = tmp / 'cat.txt'
    cl.write_text(''.join(f"file '{s}'\n" for s in (seg1, seg2, seg3)))
    run([ffmpeg, '-y', '-f', 'concat', '-safe', '0', '-i', str(cl), '-c', 'copy', str(out)])
    dur = float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', str(out)],
                               capture_output=True, text=True).stdout)
    shutil.rmtree(tmp, ignore_errors=True)
    return dur


if __name__ == '__main__':
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument('--infos', required=True)
    p.add_argument('--kitti', required=True, help='KITTI root that contains training/velodyne')
    p.add_argument('--pre', required=True, help='results folder of the pretrained model (robustness/)')
    p.add_argument('--ft', required=True, help='results folder of the fine-tuned model (robustness_finetuned/)')
    p.add_argument('--out', default='demo.mp4')
    p.add_argument('--frames', nargs='+', default=['002477', '004527', '005510', '006682'])
    p.add_argument('--rotate_frame')
    p.add_argument('--min_score', type=float, default=0.3)
    a = p.parse_args()
    with open(a.infos, 'rb') as f:
        infos = pickle.load(f)
    d = build(infos, a.kitti, a.pre, a.ft, a.out, a.frames, a.rotate_frame, a.min_score)
    print(f'wrote {a.out}: {d:.1f} s', '(OK, <= 60 s)' if d <= 60 else '(TOO LONG: use fewer --frames)')

Writing /content/OpenPCDet/tools/make_demo_video.py


In [11]:
%cd /content/OpenPCDet/tools
!python make_demo_video.py --infos ../data/kitti/kitti_infos_val.pkl --kitti ../data/kitti \
    --pre /content/drive/MyDrive/pcdet_results/robustness --ft /content/drive/MyDrive/pcdet_results/robustness_finetuned \
    --out /content/drive/MyDrive/pcdet_results/demo.mp4 --frames 002477 004527 005510 006682

/content/OpenPCDet/tools
wrote /content/drive/MyDrive/pcdet_results/demo.mp4: 34.0 s (OK, <= 60 s)


In [12]:
from IPython.display import Video
Video('/content/drive/MyDrive/pcdet_results/demo.mp4', embed=True, width=800)

Output hidden; open in https://colab.research.google.com to view.